In [1]:
import sys, os, glob, time, socket
from loguru import logger
from tabulate import tabulate
import pandas as pd
import numpy as np
import joblib   
import random

In [2]:
ANTIGENS = [
    "Diphtheria",
    "Pertussis",
    "Polio",
    "Tetanus",
    "Rotavirus",
    "PCV",
    "Measles",
    "Mumps",
    "Rubella",
    "Hepatitis_B",
    "Hib",
    "HPV",
]

N_YEARS = 10
YEARS = np.arange(1, N_YEARS + 1)

# Reformat the capacity scenarios

In [3]:
# capacity_scenario_names = [
#     "base_capacity",
#     "pandemic",
# ]
# capacity_scenario_probs = {
#     "base_capacity": 0.8418,
#     "pandemic": 0.0275,
# }

capacity_scenario_names = [
    "base_capacity",
    "IPV_Shortage",
    "pandemic",
    "funding_delay",
    "innacurate_forecast",
    "supply_chain",
    "other",
]
capacity_scenario_probs = {
    "base_capacity": 0.8418,
    "IPV_Shortage": 0.0173,
    "pandemic": 0.0275,
    "funding_delay": 0.0778,
    "innacurate_forecast": 0.0023,
    "supply_chain": 0.0058,
    "other": 0.0275,
}

In [4]:
dfs = []
for scenario in capacity_scenario_names:
    temp = pd.read_excel(
        f"data\production_capacity_scenarios.xlsx",
        sheet_name=scenario,
    )
    temp["capacity_scenario"] = scenario
    temp["capacity_scenario_probability"] = capacity_scenario_probs[scenario]
    dfs.append(temp)

print(temp)
# Concatenate all the dataframes
capacity_scenarios = pd.concat(dfs, ignore_index=True)
# rename the columns
capacity_scenarios.rename(columns={"Manufacturer": "manufacturer"}, inplace=True)
# Convert years to columns
capacity_scenarios["capacity"] = capacity_scenarios[YEARS].values.tolist()
# Drop the years columns
capacity_scenarios.drop(YEARS, axis=1, inplace=True)

capacity_scenarios.to_csv("data\production_capacity_scenarios.csv", index=False)

       Manufacturer          1          2          3          4          5  \
0       AJ_Vaccines    7711003    7711003    7094122    6939902    7479673   
1          BB_NCIPD   39147073   39223956   39223956   39120138   39223956   
2    Bharat_Biotech   61029105   61029105   61029105   61029105   61029105   
3         Bilthoven   12048153   12048153   11084301   10843338   11686709   
4      Biological_E  164256564  164885690  164256564  164544872  164885690   
5    China_National   12812242   12627715   12812242   12812242   12812242   
6               GSK  226762686  226380511  226762686  223728150  226762686   
7      Haffkine_Bio   80972855   80972855   80972855   80972855   80972855   
8           LG_Chem   43536522   43702188   43493336   43441123   43623868   
9       Merck_Sharp   56991052   52517254   56991052   56991052   56991052   
10           PT_Bio   45864272   45911731   45911731   45702427   45436226   
11   Panacea_Biotec   10874165   10874165   10874165   10874165 

In [5]:
capacity_scenarios

,manufacturer,capacity_scenario,capacity_scenario_probability,capacity
0,AJ_Vaccines,base_capacity,0.8418,"[7711003.0, 7711003.0, 7711003.0, 7711003.0, 7..."
1,BB_NCIPD,base_capacity,0.8418,"[39223956.0, 39223956.0, 39223956.0, 39223956...."
2,Bharat_Biotech,base_capacity,0.8418,"[61029105.0, 61029105.0, 61029105.0, 61029105...."
3,Bilthoven,base_capacity,0.8418,"[12048153.0, 12048153.0, 12048153.0, 12048153...."
4,Biological_E,base_capacity,0.8418,"[164885690.0, 164885690.0, 164885690.0, 164885..."
...,...,...,...,...
100,PT_Bio,other,0.0275,"[45864272.0, 45911731.0, 45911731.0, 45702427...."
101,Panacea_Biotec,other,0.0275,"[10874165.0, 10874165.0, 10874165.0, 10874165...."
102,Pfizer,other,0.0275,"[83916974.0, 83916974.0, 83916974.0, 79721125...."
103,Sanofi,other,0.0275,"[87429432.0, 87429432.0, 82229737.0, 80966413...."


In [6]:
import pandas as pd
import numpy as np

# Load data
demand_scenarios_df = pd.read_csv(
    "data/OOB/antigen_demand_80_20_5_scenarios.csv",
    converters={"demands": pd.eval},
    usecols=["demands", "antigen", "demand_SID", "prob"],
)

# capacity_scenarios_df = pd.read_csv(
#     "data/production_capacity_scenarios.csv",
#     converters={"capacity": pd.eval},
# )

capacity_scenarios_df = capacity_scenarios

# Function to generate pairs
def generate_pairs(demand: pd.DataFrame, capacity: pd.DataFrame, n_pairs: int = 10, verbose: bool = True):
    demand_dict = demand.drop_duplicates(subset=["demand_SID"]).set_index("demand_SID")["prob"].to_dict()
    capacity_dict = capacity.drop_duplicates(subset=["capacity_scenario"]).set_index("capacity_scenario")["capacity_scenario_probability"].to_dict()

    demand_keys = list(demand_dict.keys())
    demand_probs = list(demand_dict.values())

    capacity_keys = list(capacity_dict.keys())
    capacity_probs = list(capacity_dict.values())

    if verbose:
        print(f"Unique demand scenarios: {demand_keys}")
        print(f"Unique capacity scenarios: {capacity_keys}")

    pairs = []
    pair_dfs = []
    prob_dict = {}
    pair_idx = 1

    for selected_capacity, selected_prob_capacity in capacity_dict.items():
        available_demand_keys = demand_keys.copy()
        available_demand_probs = demand_probs.copy()

        for _ in range(n_pairs):
            # Normalize the probabilities
            available_demand_probs = [p / sum(available_demand_probs) for p in available_demand_probs]

            selected_demand = np.random.choice(available_demand_keys, p=available_demand_probs)
            # Combine the probabilities
            selected_prob_demand = demand_dict[selected_demand]

            combined_prob = selected_prob_demand * selected_prob_capacity
            prob_dict[pair_idx] = combined_prob

            # Get the selected demand and capacity scenarios
            selected_demand_df = demand[demand["demand_SID"] == selected_demand].copy()
            selected_demand_df["type"] = "antigen"
            selected_demand_df.drop(columns=["prob", "demand_SID"], inplace=True)
            selected_demand_df.rename(columns={"antigen": "unit", "demands": "values"}, inplace=True)

            selected_capacity_df = capacity[capacity["capacity_scenario"] == selected_capacity].copy()
            selected_capacity_df["type"] = "manufacturer"
            selected_capacity_df.drop(columns=["capacity_scenario_probability", "capacity_scenario"], inplace=True)
            selected_capacity_df.rename(columns={"manufacturer": "unit", "capacity": "values"}, inplace=True)

            pair_df = pd.concat([selected_demand_df, selected_capacity_df])

            # Add additional information
            pair_df["pair_idx"] = pair_idx
            pair_df["pair"] = f"Demand: {selected_demand} - Capacity: {selected_capacity}"
            pair_dfs.append(pair_df)
            pairs.append((selected_demand, selected_capacity))
            pair_idx += 1

            # Remove the selected demand from the available demands
            index = available_demand_keys.index(selected_demand)
            available_demand_keys.pop(index)
            available_demand_probs.pop(index)

    pair_df = pd.concat(pair_dfs, ignore_index=True)

    # Calculate the sum of all probabilities
    total_sum = sum(prob_dict.values())

    # Scale the probabilities so they sum up to 1
    scaled_probabilities = {k: v / total_sum for k, v in prob_dict.items()}
    pair_df["pair_probability"] = pair_df["pair_idx"].map(scaled_probabilities)

    return pairs, pair_df

# Generate pairs
scenario_pairs, pair_df = generate_pairs(demand_scenarios_df, capacity_scenarios_df, n_pairs=5, verbose=True)

# Export to json and csv
pair_df.to_json("data/OOB/pair_demand_capacity_new.json", orient="records", lines=True)
pair_df.to_csv("data/OOB/pair_demand_capacity_new.csv", index=False)

# Display the DataFrame
pair_df


Unique demand scenarios: [2, 3, 4, 5, 6]
Unique capacity scenarios: ['base_capacity', 'IPV_Shortage', 'pandemic', 'funding_delay', 'innacurate_forecast', 'supply_chain', 'other']


,unit,values,type,pair_idx,pair,pair_probability
0,Diphtheria,"[670487400, 735223400, 677717200, 802467500, 9...",antigen,1,Demand: 4 - Capacity: base_capacity,0.10775
1,HPV,"[30334800, 35937000, 39631100, 46530800, 51332...",antigen,1,Demand: 4 - Capacity: base_capacity,0.10775
2,Hepatitis_B,"[370407500, 444508300, 399556600, 426122800, 4...",antigen,1,Demand: 4 - Capacity: base_capacity,0.10775
3,Hib,"[281761500, 293066700, 346902200, 330750100, 3...",antigen,1,Demand: 4 - Capacity: base_capacity,0.10775
4,Measles,"[353668900, 489897600, 638311200, 403718000, 6...",antigen,1,Demand: 4 - Capacity: base_capacity,0.10775
...,...,...,...,...,...,...
940,PT_Bio,"[45864272.0, 45911731.0, 45911731.0, 45702427....",manufacturer,35,Demand: 4 - Capacity: other,0.00352
941,Panacea_Biotec,"[10874165.0, 10874165.0, 10874165.0, 10874165....",manufacturer,35,Demand: 4 - Capacity: other,0.00352
942,Pfizer,"[83916974.0, 83916974.0, 83916974.0, 79721125....",manufacturer,35,Demand: 4 - Capacity: other,0.00352
943,Sanofi,"[87429432.0, 87429432.0, 82229737.0, 80966413....",manufacturer,35,Demand: 4 - Capacity: other,0.00352


# Create scenario pairs

In [7]:
pair_df.set_index("pair_idx")

,unit,values,type,pair,pair_probability
pair_idx,,,,,
1,Diphtheria,"[670487400, 735223400, 677717200, 802467500, 9...",antigen,Demand: 4 - Capacity: base_capacity,0.10775
1,HPV,"[30334800, 35937000, 39631100, 46530800, 51332...",antigen,Demand: 4 - Capacity: base_capacity,0.10775
1,Hepatitis_B,"[370407500, 444508300, 399556600, 426122800, 4...",antigen,Demand: 4 - Capacity: base_capacity,0.10775
1,Hib,"[281761500, 293066700, 346902200, 330750100, 3...",antigen,Demand: 4 - Capacity: base_capacity,0.10775
1,Measles,"[353668900, 489897600, 638311200, 403718000, 6...",antigen,Demand: 4 - Capacity: base_capacity,0.10775
...,...,...,...,...,...
35,PT_Bio,"[45864272.0, 45911731.0, 45911731.0, 45702427....",manufacturer,Demand: 4 - Capacity: other,0.00352
35,Panacea_Biotec,"[10874165.0, 10874165.0, 10874165.0, 10874165....",manufacturer,Demand: 4 - Capacity: other,0.00352
35,Pfizer,"[83916974.0, 83916974.0, 83916974.0, 79721125....",manufacturer,Demand: 4 - Capacity: other,0.00352


In [10]:
pair_df.loc[1]

unit                                                              HPV
values              [18988200, 18779800, 27460000, 72849500, 92670...
type                                                          antigen
pair_idx                                                            1
pair                              Demand: 1 - Capacity: base_capacity
pair_probability                                              0.67344
Name: 1, dtype: object

In [11]:
pair_df['unit'].unique()

array(['Diphtheria', 'HPV', 'Hepatitis_B', 'Hib', 'Measles', 'Mumps',
       'PCV', 'Pertussis', 'Polio', 'Rotavirus', 'Rubella', 'Tetanus',
       'AJ_Vaccines', 'BB_NCIPD', 'Bharat_Biotech', 'Bilthoven',
       'Biological_E', 'China_National', 'GSK', 'Haffkine_Bio', 'LG_Chem',
       'Merck_Sharp', 'PT_Bio', 'Panacea_Biotec', 'Pfizer', 'Sanofi',
       'Serum_Institute'], dtype=object)

In [12]:
pair_df

,unit,values,type,pair_idx,pair,pair_probability
0,Diphtheria,"[570550200, 619539900, 626455400, 594496900, 5...",antigen,1,Demand: 1 - Capacity: base_capacity,0.673440
1,HPV,"[18988200, 18779800, 27460000, 72849500, 92670...",antigen,1,Demand: 1 - Capacity: base_capacity,0.673440
2,Hepatitis_B,"[312284600, 343008200, 351717800, 334973600, 3...",antigen,1,Demand: 1 - Capacity: base_capacity,0.673440
3,Hib,"[259536200, 282777100, 287252900, 274027900, 2...",antigen,1,Demand: 1 - Capacity: base_capacity,0.673440
4,Measles,"[420097000, 358616900, 394831600, 346093200, 3...",antigen,1,Demand: 1 - Capacity: base_capacity,0.673440
...,...,...,...,...,...,...
940,PT_Bio,"[45864272.0, 45911731.0, 45911731.0, 45702427....",manufacturer,35,Demand: 2 - Capacity: other,0.000517
941,Panacea_Biotec,"[10874165.0, 10874165.0, 10874165.0, 10874165....",manufacturer,35,Demand: 2 - Capacity: other,0.000517
942,Pfizer,"[83916974.0, 83916974.0, 83916974.0, 79721125....",manufacturer,35,Demand: 2 - Capacity: other,0.000517
943,Sanofi,"[87429432.0, 87429432.0, 82229737.0, 80966413....",manufacturer,35,Demand: 2 - Capacity: other,0.000517


In [9]:
pair_df.groupby('unit').count()

,values,type,pair_idx,pair,pair_probability
unit,,,,,
AJ_Vaccines,35,35,35,35,35
BB_NCIPD,35,35,35,35,35
Bharat_Biotech,35,35,35,35,35
Bilthoven,35,35,35,35,35
Biological_E,35,35,35,35,35
China_National,35,35,35,35,35
Diphtheria,35,35,35,35,35
GSK,35,35,35,35,35
HPV,35,35,35,35,35
